### USGS Earthquake FDSNWS API analysis

This short notebook showcases the REST API used to ingest Earthquake data into Snowflake. A similar approach will be used in stored procedures.

The API is publically available so no authorisation is required.

In [1]:
import requests
import pandas as pd

URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",
    "starttime": "2026-01-01",
    "minmagnitude": 5
}

response = requests.get(URL, params=params)
response.raise_for_status()  # raises an error if the request failed
data = response.json()

In [2]:
# Each earthquake is a "feature" with properties + geometry (coordinates)
records = []
for feature in data["features"]:
    props = feature["properties"]
    coords = feature["geometry"]["coordinates"]  # [longitude, latitude, depth]
    records.append({
        "id": feature["id"],
        "place": props.get("place"),
        "magnitude": props.get("mag"),
        "time": pd.to_datetime(props.get("time"), unit="ms"),
        "updated": pd.to_datetime(props.get("updated"), unit="ms"),
        "tsunami": props.get("tsunami"),
        "type": props.get("type"),
        "longitude": coords[0],
        "latitude": coords[1],
        "depth_km": coords[2],
    })

In [3]:
df = pd.DataFrame(records)
print(f"Retrieved {len(df)} earthquakes")

df.head()

Retrieved 945 earthquakes


,id,place,magnitude,time,updated,tsunami,type,longitude,latitude,depth_km
0,us6000taft,"211 km ESE of Hihifo, Tonga",5.1,2026-07-06 14:43:38.525,2026-07-06 15:16:56.040,0,earthquake,-172.0700,-16.8920,10.0
1,us6000tadj,"65 km SSW of Sarangani, Philippines",5.4,2026-07-06 08:41:38.715,2026-07-06 15:10:24.009,0,earthquake,125.1424,4.9091,35.0
2,us6000tadc,"48 km SSW of Sarangani, Philippines",5.8,2026-07-06 08:11:11.032,2026-07-06 15:07:25.422,0,earthquake,125.3260,4.9886,35.0
3,us6000taab,"106 km NW of Coquimbo, Chile",5.0,2026-07-05 15:53:33.718,2026-07-05 16:09:50.040,0,earthquake,-72.0113,-29.1964,10.0
4,us6000taaa,"26 km NNW of Mianzhu, Deyang, Sichuan, China",5.0,2026-07-05 15:30:13.631,2026-07-05 23:45:29.794,0,earthquake,104.1260,31.5649,10.0
